# 端到端迁移教学版

这个 notebook 对应第 10 章，用来演示如何把前面的算子优化、跨平台迁移和 benchmark 统一到一个项目流程里。

它关注的是流程，不是某个单点实现。

## 阅读顺序

1. 先看为什么选子图，而不是完整模型
2. 再看算子清单和优先级
3. 然后看第一个可迁移算子如何形成闭环
4. 再看逐步优化和多后端适配
5. 最后看端到端 benchmark 记录方式

## 迁移场景骨架

- 一个明确的模型子图
- 一组稳定的输入输出形状
- 一个可比对的 baseline
- 一个或多个目标后端
- 一套统一的验证和 benchmark 方法

In [ ]:
migration_plan = [
    '1. choose a stable hotspot subgraph',
    '2. list operators and rank them',
    '3. implement the first operator and close the loop',
    '4. iterate on optimization in small steps',
    '5. adapt to multiple backends',
    '6. record operator-level, subgraph-level, and end-to-end benchmark data',
]

for step in migration_plan:
    print(step)

## 第一个可迁移算子

最适合先做的通常是 `GEMM`，因为它稳定、可验证，而且能代表很多后续热点算子的共性。

最小闭环是：

1. baseline 跑通
2. TileLang 版本跑通
3. 正确性对齐
4. benchmark 补齐

## 迭代优化记录

每一轮都应该固定记录：

- 改动点
- 为什么改
- 改完后的收益
- 引入的新风险
- 是否值得保留

In [ ]:
benchmark_record = {
    'scenario': 'end-to-end migration',
    'operator': 'subgraph benchmark',
    'platform': 'multi-backend',
    'target': 'cuda',
    'version': 'draft',
    'driver_version': 'unknown',
    'runtime_version': 'unknown',
    'dtype': 'float16',
    'shape': 'subgraph specific',
    'batch': 1,
    'warmup': 10,
    'runs': 100,
    'latency_ms': None,
    'throughput': None,
    'tflops': None,
    'gbps': None,
    'peak_mem_mb': None,
    'avg_mem_mb': None,
    'baseline': 'original implementation',
    'optimization': 'iterative optimization',
    'notes': 'fill after each migration step',
}

for key in ['scenario', 'operator', 'platform', 'target', 'dtype', 'warmup', 'runs']:
    print(f'{key}: {benchmark_record[key]}')

## 端到端 benchmark 的三层记录

- 算子级
- 子图级
- 端到端级

先把这三层分开，再谈跨后端比较。

## 收口方式

一份端到端项目最终应该能回答：

- 迁移了什么
- 为什么先迁移这些算子
- 哪些优化有效
- benchmark 提供了什么证据